# LUCID on Bike Sharing

Fit an XGBoost predictor, then explain it using TabPFN 3.5-Fast conditional sampling.
Install LUCID with the `xgboost` and `tabpfn` extras, and make Jupyter available in the same environment.
Local TabPFN requires access to its pretrained checkpoints.

The training features serve as the background. We explain 50 held-out observations.
The sampler uses its default local backend and automatic device selection.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

import lucid


X, y = lucid.datasets.bikesharing()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2026,
)
model = XGBRegressor(random_state=2026)
model.fit(X_train, y_train)
X_eval = X_test.iloc[:50]

feature_types = [
    "categorical" if name in {"hr", "season"}
    else "binary" if name == "workingday"
    else "continuous"
    for name in X.columns
]


## Feature LUCID

The predictive model stays fixed. LUCID measures the expected squared prediction change after conditionally replacing a feature.
The default number of replacement draws is 1,000. `n_jobs=2` allows two feature tasks at a time.


In [ ]:
explainer = lucid.Explainer(
    model,
    X_train,
    sampler="tabpfn",
    feature_types=feature_types,
    random_state=2026,
    n_jobs=2,
    sampler_kwargs={"version": "v3.5-fast"},
)
try:
    explanation = explainer(X_eval)
finally:
    explainer.close()

print("LUCID values:", explanation.values.shape)
print("Global LUCID:", explanation.global_importance())


## Plots

Each explanation row corresponds to the same row in `X_eval`.
Use `cmap` to choose a Matplotlib colormap for feature values.


In [ ]:
lucid.plots.bar(explanation)
plt.show()


In [ ]:
lucid.plots.beeswarm(
    explanation, X_eval,
    categorical=["hr", "workingday", "season"],
    cmap="viridis",
)
plt.show()


In [ ]:
lucid.plots.scatter(
    explanation, X_eval, feature="hum", color="temp", cmap="viridis",
)
plt.show()


The relationship plot shows Pearson correlations between numeric features alongside LUCID values. It does not estimate conditional predictability. Nominal features are excluded from the correlation links.


In [ ]:
lucid.plots.relationship(
    explanation, X_eval,
    categorical=["hr", "workingday", "season"],
    cmap="viridis",
)
plt.show()


In [ ]:
# Zero-based observation positions, with four panels per row.
lucid.plots.observations(
    explanation, observations=range(8), ncols=4,
)
plt.show()


## Group LUCID

Replace a group's features jointly. Member order determines the conditional sampling order.
These values are calculated directly for each group; they are not sums of individual feature values.


In [ ]:
group_explainer = lucid.GroupExplainer(
    model,
    X_train,
    sampler="tabpfn",
    feature_types=feature_types,
    random_state=2026,
    n_jobs=2,
    sampler_kwargs={"version": "v3.5-fast"},
)
try:
    group_explanation = group_explainer(
        X_eval,
        groups={
            "weather": ["temp", "hum"],
            "calendar": ["hr", "workingday", "season"],
        },
    )
finally:
    group_explainer.close()

print("Group LUCID values:", group_explanation.values.shape)
print("Global group LUCID:", group_explanation.global_importance())


In [ ]:
lucid.plots.bar(group_explanation)
lucid.plots.observations(
    group_explanation, observations=range(4), ncols=4,
)
plt.show()


For DRBART posterior uncertainty on Auto MPG, see `uncertainty.py`.
Those intervals reflect uncertainty in the fitted conditional model while holding the predictive model fixed; finite resampling also introduces Monte Carlo variation.
